# AI-Based Road Debris & Obstacle Detection System

This Google Colab notebook implements a complete, self-contained **Road Debris & Obstacle Detection System** using **YOLOv8** and custom class-aware IoU tracking.

### Key Features:
1. **Object Detection**: Identifies vehicles, pedestrians, and potential debris (like suitcases, bags, and chairs) using pre-trained YOLOv8 models.
2. **Class-Aware Tracking**: Tracks multiple objects across frames using a custom IoU-based tracking algorithm, maintaining consistent IDs.
3. **Region of Interest (ROI) Filtering**: Restricts detections to the active roadway using a configurable road polygon to prevent false alerts from off-road objects.
4. **Stationary Obstacle Heuristics**: Detects stopped objects on the road by checking displacement over time. Integrates dual-persistence thresholds (debris is flagged quickly, whereas normal vehicles/pedestrians are flagged only after sustained immobility).
5. **Logging & Alerting**: Saves evidence snapshots, prints real-time alerts, and maintains a structured CSV incident report.
6. **Analytics Dashboard**: Dynamically visualizes session statistics and maps incident timelines.

---


## 1. Setup & Environment Configurations

In this section, we install the required libraries, mount Google Drive (optional) to save outputs permanently, create the directory structure, and load the pre-trained YOLOv8 weights.

In [ ]:
# Install required libraries
!pip install -q ultralytics opencv-python-headless pandas matplotlib

import os
import torch

# PyTorch 2.6 compatibility patch for loading YOLO models safely
try:
    _orig_load = torch.load
    def _patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig_load(*args, **kwargs)
    torch.load = _patched_load
except Exception:
    pass

from google.colab import drive
from ultralytics import YOLO

# 1. Mount Google Drive
try:
    drive.mount('/content/drive')
    print("[OK] Google Drive mounted successfully.")
    USE_DRIVE = True
except Exception as e:
    print(f"[WARN] Google Drive mount skipped: {e}")
    print("Using local directory structure instead.")
    USE_DRIVE = False

# 2. Setup Folder Structure
if USE_DRIVE:
    BASE_DIR = "/content/drive/MyDrive/Task138_RoadDebrisDetectionSystem"
else:
    BASE_DIR = "./Task138_RoadDebrisDetectionSystem"

INPUT_DIR = os.path.join(BASE_DIR, "Inputs")
EVIDENCE_DIR = os.path.join(BASE_DIR, "Outputs", "evidence_frames")
LOGS_DIR = os.path.join(BASE_DIR, "Outputs")

os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(EVIDENCE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print("\nFolder structure initialized:")
print(f" - Inputs Directory:  {INPUT_DIR}")
print(f" - Evidence Snapshot: {EVIDENCE_DIR}")
print(f" - Incident Logs:     {LOGS_DIR}")

# 3. Load YOLOv8 Model
print("\nLoading YOLOv8-detection model...")
model = YOLO("yolov8n.pt")
print("[OK] YOLOv8 model loaded successfully.")

## 2. Video & Image Sequence Input Generator

We define an efficient frame reader supporting standard videos (`.mp4`, `.avi`, etc.) and directories containing sequential images (common in driving research datasets).

In [ ]:
import cv2
import glob

def video_frame_generator(video_path):
    """
    Opens a video file OR a directory containing sequential images, and yields frames 
    sequentially along with sequence metadata.
    """
    if not os.path.exists(video_path):
        raise FileNotFoundError(f"Input path not found at: {video_path}")
        
    if os.path.isdir(video_path):
        image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.bmp', '*.tif', '*.tiff')
        image_files = []
        for ext in image_extensions:
            image_files.extend(glob.glob(os.path.join(video_path, ext)))
            image_files.extend(glob.glob(os.path.join(video_path, ext.upper())))
            
        image_files = sorted(list(set(image_files)))
        
        if not image_files:
            raise FileNotFoundError(f"No image files found in directory: {video_path}")
            
        frame_count = len(image_files)
        first_frame = cv2.imread(image_files[0])
        if first_frame is None:
            raise IOError(f"Could not read the first image frame: {image_files[0]}")
        height, width = first_frame.shape[:2]
        fps = 30.0
        
        for frame_idx, img_path in enumerate(image_files):
            frame = cv2.imread(img_path)
            if frame is None:
                print(f"⚠️ Warning: Could not read frame image: {img_path}")
                continue
            yield frame, frame_idx, fps, frame_count, width, height
            
    else:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            raise IOError(f"OpenCV was unable to open the video file at: {video_path}")
            
        try:
            fps = cap.get(cv2.CAP_PROP_FPS)
            if fps <= 0 or fps is None:
                fps = 30.0
                
            frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
            height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
            
            frame_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                yield frame, frame_idx, fps, frame_count, width, height
                frame_idx += 1
        finally:
            cap.release()

## 3. Class-Aware Tracking & ROI checking

We implement standard IoU math, the custom class-aware tracker `SimpleIoUTracker` that filters targets and maps centroids over time, and a ray-casting method `is_inside_polygon` to verify if objects are inside the roadway lanes.

In [ ]:
import numpy as np

def calculate_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    if x2 < x1 or y2 < y1:
        return 0.0
        
    intersection_area = (x2 - x1) * (y2 - y1)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area
    
    if union_area == 0.0:
        return 0.0
        
    return intersection_area / union_area

class SimpleIoUTracker:
    def __init__(self, iou_threshold=0.35, max_lost_frames=30):
        self.iou_threshold = iou_threshold
        self.max_lost_frames = max_lost_frames
        self.next_id = 1
        self.tracked_objects = {}
        
    def update(self, detections, frame_idx):
        active_ids = list(self.tracked_objects.keys())
        matches = []
        
        for det_idx, det in enumerate(detections):
            det_bbox = det["bbox"]
            det_cls = det["class_id"]
            for track_id in active_ids:
                track_data = self.tracked_objects[track_id]
                if track_data["class_id"] == det_cls:
                    track_bbox = track_data["bbox"]
                    iou = calculate_iou(det_bbox, track_bbox)
                    if iou >= self.iou_threshold:
                        matches.append((iou, det_idx, track_id))
                        
        matches.sort(key=lambda x: x[0], reverse=True)
        matched_det_indices = set()
        matched_track_ids = set()
        
        for iou, det_idx, track_id in matches:
            if det_idx in matched_det_indices or track_id in matched_track_ids:
                continue
                
            matched_det_indices.add(det_idx)
            matched_track_ids.add(track_id)
            
            det = detections[det_idx]
            track_data = self.tracked_objects[track_id]
            track_data["bbox"] = det["bbox"]
            track_data["confidence"] = det["confidence"]
            track_data["lost_frames"] = 0
            track_data["last_seen_frame"] = frame_idx
            
            centroid_x = (det["bbox"][0] + det["bbox"][2]) / 2.0
            centroid_y = (det["bbox"][1] + det["bbox"][3]) / 2.0
            track_data["centroid_history"].append((centroid_x, centroid_y, frame_idx))
            
            if len(track_data["centroid_history"]) > 150:
                track_data["centroid_history"].pop(0)
                
        for det_idx, det in enumerate(detections):
            if det_idx not in matched_det_indices:
                centroid_x = (det["bbox"][0] + det["bbox"][2]) / 2.0
                centroid_y = (det["bbox"][1] + det["bbox"][3]) / 2.0
                
                self.tracked_objects[self.next_id] = {
                    "bbox": det["bbox"],
                    "confidence": det["confidence"],
                    "class_id": det["class_id"],
                    "class_name": det["class_name"],
                    "centroid_history": [(centroid_x, centroid_y, frame_idx)],
                    "lost_frames": 0,
                    "first_seen_frame": frame_idx,
                    "last_seen_frame": frame_idx,
                    "stationary_frames": 0,
                    "is_hazard": False,
                    "hazard_logged": False
                }
                self.next_id += 1
                
        dead_tracks = []
        for track_id in active_ids:
            if track_id not in matched_track_ids:
                self.tracked_objects[track_id]["lost_frames"] += 1
                if self.tracked_objects[track_id]["lost_frames"] > self.max_lost_frames:
                    dead_tracks.append(track_id)
                    
        for track_id in dead_tracks:
            del self.tracked_objects[track_id]
            
        return {
            tid: data for tid, data in self.tracked_objects.items() 
            if data["last_seen_frame"] == frame_idx
        }

def is_inside_polygon(point, polygon):
    poly_arr = np.array(polygon, dtype=np.float32)
    pt = (float(point[0]), float(point[1]))
    result = cv2.pointPolygonTest(poly_arr, pt, False)
    return result >= 0

## 4. Incident Logger & Alerting

We log incidents on transition to active hazards, print warning messages, and write evidence snapshots.

In [ ]:
import datetime
import pandas as pd

def log_debris_incident(frame, frame_idx, track_id, track_data, video_name, evidence_dir, log_list):
    cls_name = track_data["class_name"]
    conf = track_data["confidence"]
    
    print(f"⚠️ [ALERT] STATIONARY ROAD OBSTACLE: {cls_name.upper()} (ID {track_id}) detected at frame {frame_idx}")
    
    timestamp_str = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    evidence_filename = f"incident_{timestamp_str}_{cls_name}_id_{track_id}.jpg"
    evidence_path = os.path.join(evidence_dir, evidence_filename)
    
    cv2.imwrite(evidence_path, frame)
    
    log_list.append({
        "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "event_id": f"{os.path.splitext(video_name)[0]}_id_{track_id}",
        "object_id": track_id,
        "class_name": cls_name,
        "confidence_score": float(conf),
        "frame_number": frame_idx,
        "evidence_filename": evidence_filename,
        "video_name": video_name,
        "status": "STATIONARY_OBSTACLE"
    })
    
    return evidence_filename

## 5. Analytics Dashboard

This cell reads the generated CSV log and visualizes incident metrics.

In [ ]:
import json
import matplotlib.pyplot as plt

def display_analytics_dashboard(logs_dir):
    csv_path = os.path.join(logs_dir, "incident_log.csv")
    summary_path = os.path.join(logs_dir, "summary_stats.json")
    
    if not os.path.exists(csv_path):
        print("❌ No incident log CSV found. Process a video first.")
        return
        
    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"❌ Error loading incident log CSV: {e}")
        return
        
    print("\n" + "="*60)
    print("             ROAD DEBRIS DETECTION SYSTEM ANALYTICS")
    print("="*60)
    
    if os.path.exists(summary_path):
        try:
            with open(summary_path, 'r') as f:
                stats = json.load(f)
            print(f"Total Incidents Logged:        {stats.get('total_incidents', 0)}")
            print(f"Unique Obstacles Tracked:      {stats.get('unique_obstacles_tracked', 0)}")
            print(f"Average Detection Confidence:  {stats.get('avg_confidence', 0.0):.2f}")
            print(f"Most Frequent Hazard Class:    {stats.get('most_frequent_hazard_class', 'N/A')}")
        except Exception as e:
            print(f"Warning: Could not read summary_stats.json: {e}")
    else:
        print(f"Total Incidents Logged:        {len(df)}")
        if len(df) > 0:
            print(f"Unique Obstacles Tracked:      {df['object_id'].nunique()}")
            print(f"Average Detection Confidence:  {df['confidence_score'].mean():.2f}")
            print(f"Most Frequent Hazard Class:    {df['class_name'].mode()[0] if not df['class_name'].empty else 'N/A'}")
            
    print("="*60 + "\n")
    
    if len(df) == 0:
        print("Zero incidents logged. Skipping dashboard plot rendering.")
        return
        
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    secondary_color = '#4A90E2'
    
    class_counts = df['class_name'].value_counts()
    axes[0].bar(class_counts.index, class_counts.values, color=secondary_color, edgecolor='black', zorder=2)
    axes[0].set_title("Incident Alert Count by Obstacle/Debris Class", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Object Class", fontsize=10)
    axes[0].set_ylabel("Number of Incidents", fontsize=10)
    axes[0].tick_params(axis='x', rotation=30)
    axes[0].grid(axis='y', linestyle='--', alpha=0.5, zorder=1)
    
    scatter = axes[1].scatter(
        df['frame_number'], 
        df['class_name'].astype(str) + " (ID " + df['object_id'].astype(str) + ")",
        s=df['confidence_score'] * 350,
        c=df['confidence_score'],
        cmap='plasma',
        edgecolors='black',
        alpha=0.85,
        zorder=2
    )
    axes[1].set_title("Incident Timeline (Frame vs. Obstacle ID)", fontsize=12, fontweight='bold')
    axes[1].set_xlabel("Frame Number", fontsize=10)
    axes[1].set_ylabel("Hazard Type & ID", fontsize=10)
    axes[1].grid(True, linestyle='--', alpha=0.5, zorder=1)
    
    cbar = fig.colorbar(scatter, ax=axes[1])
    cbar.set_label('YOLOv8 Detection Confidence Score', fontsize=9)
    
    plt.tight_layout()
    plt.show()

## 6. End-to-End Pipeline Execution

This final cell runs the complete pipeline on your test video. Edit the config parameters below to run on your files.

In [ ]:
import time
import json

# =====================================================================
# CONFIGURATION - EDIT TO PROCESS YOUR VIDEOS
# =====================================================================
# Place your video inside Task138_RoadDebrisDetectionSystem/Inputs/
TEST_VIDEO = os.path.join(INPUT_DIR, "trash.mp4")
ANNOTATED_OUT = os.path.join(BASE_DIR, "Outputs", "trash_annotated.mp4")

# Road ROI Vertices (trapezoid outline: x1,y1,x2,y2,x3,y3,x4,y4 normalized)
ROAD_ROI_STR = "0.05,0.95,0.40,0.50,0.60,0.50,0.95,0.95"

# Persistence settings (in seconds)
DEBRIS_PERSISTENCE = 1.5
VEHICLE_PERSISTENCE = 4.0

# Stationary movement threshold (fraction of video width)
STATIONARY_THRESH = 0.015
CAMERA_MOTION = "moving"  # stationary or moving camera
MOVING_PERSISTENCE = 0.2  # persistence in seconds for moving mode
SPEED_THRESHOLD = 0.002   # velocity threshold for approaching objects
# =====================================================================

def run_debris_detection_pipeline(video_path, output_video_path):
    if not os.path.exists(video_path):
        print(f"❌ Error: Video not found at: {video_path}")
        return
        
    print(f"🎬 Initializing pipeline for: {video_path}")
    start_time = time.time()
    processed_frames = 0
    total_tracked_hazards = set()
    incident_logs = []
    
    debris_classes = ["backpack","umbrella","handbag","suitcase","bottle","cup","chair","couch","potted plant","sports ball","frisbee","stop sign","book","box"]
    vehicle_classes = ["person","car","motorcycle","bus","truck","bicycle","dog","cat","horse","cow","sheep"]
    all_allowed = set(debris_classes + vehicle_classes)
    
    roi_floats = [float(x) for x in ROAD_ROI_STR.split(',')]
    normalized_roi = [(roi_floats[i], roi_floats[i+1]) for i in range(0, len(roi_floats), 2)]
    
    tracker = SimpleIoUTracker(iou_threshold=0.35, max_lost_frames=30)
    video_filename = os.path.basename(video_path.rstrip('/\\'))
    
    frame_generator = video_frame_generator(video_path)
    out_writer = None
    
    for frame, frame_idx, fps, frame_count, width, height in frame_generator:
        if out_writer is None:
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            out_writer = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))    
            print(f"Video: {width}x{height} | {fps} FPS | {frame_count} frames")
            pixel_roi = [(int(pt[0] * width), int(pt[1] * height)) for pt in normalized_roi]
            pixel_roi_np = np.array(pixel_roi, dtype=np.int32)
            
        results = model.predict(frame, verbose=False)
        result = results[0]
        
        detections = []
        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()
            names = result.names
            
            for idx in range(len(boxes)):
                class_name = names[int(classes[idx])].lower()
                if class_name in all_allowed and confs[idx] >= 0.3:
                    detections.append({
                        "bbox": list(boxes[idx]),
                        "confidence": float(confs[idx]),
                        "class_id": int(classes[idx]),
                        "class_name": class_name
                    })
                    
        current_tracks = tracker.update(detections, frame_idx)
        annotated_frame = frame.copy()
        
        # Draw ROI overlay
        overlay = annotated_frame.copy()
        cv2.fillPoly(overlay, [pixel_roi_np], (255, 100, 0))
        cv2.addWeighted(overlay, 0.15, annotated_frame, 0.85, 0, annotated_frame)
        cv2.polylines(annotated_frame, [pixel_roi_np], True, (255, 120, 0), 2, lineType=cv2.LINE_AA)
        
        active_hazards_in_frame = []
        
        for track_id, track in current_tracks.items():
            history = track["centroid_history"]
            curr_cx, curr_cy, _ = history[-1]
            in_roi = is_inside_polygon((curr_cx, curr_cy), pixel_roi)
            
            is_stationary = False
            is_approaching = False
            
            if in_roi and len(history) >= 5:
                if CAMERA_MOTION == "moving":
                    # In moving mode, check vertical speed
                    window_history = history[-5:]
                    curr_cx, curr_cy, curr_f = window_history[-1]
                    prev_cx, prev_cy, prev_f = window_history[0]
                    fdiff = curr_f - prev_f
                    if fdiff > 0:
                        vy = (curr_cy - prev_cy) / fdiff
                        pixel_speed_thresh = SPEED_THRESHOLD * height
                        if vy >= pixel_speed_thresh:
                            is_approaching = True
                else:
                    rolling_window = max(5, int(fps * 1.5))
                    window_history = history[-rolling_window:]
                    displacements = [
                        np.sqrt((curr_cx - h_cx)**2 + (curr_cy - h_cy)**2)
                        for h_cx, h_cy, _ in window_history
                    ]
                    max_displacement = max(displacements)
                    if max_displacement <= (STATIONARY_THRESH * width):
                        is_stationary = True
            
            if in_roi and (is_stationary or is_approaching):
                track["stationary_frames"] += 1
            else:
                track["stationary_frames"] = 0
                track["is_hazard"] = False
                
            duration = track["stationary_frames"] / fps
            cls_name = track["class_name"]
            
            if CAMERA_MOTION == "moving":
                required_persistence = 0.1 if cls_name in debris_classes else MOVING_PERSISTENCE
            else:
                required_persistence = DEBRIS_PERSISTENCE if cls_name in debris_classes else VEHICLE_PERSISTENCE
            
            if track["stationary_frames"] > 0 and duration >= required_persistence:
                was_hazard = track["is_hazard"]
                track["is_hazard"] = True
                total_tracked_hazards.add(track_id)
                active_hazards_in_frame.append((track_id, cls_name))
                
                if not was_hazard and not track["hazard_logged"]:
                    log_debris_incident(frame, frame_idx, track_id, track, video_filename, EVIDENCE_DIR, incident_logs)
                    track["hazard_logged"] = True
                    
            x1, y1, x2, y2 = map(int, track["bbox"])
            if track["is_hazard"]:
                box_color, box_thick, status_lbl = (0, 0, 255), 3, "HAZARD: OBSTACLE"
            elif track["stationary_frames"] > 0:
                box_color, box_thick, status_lbl = (0, 255, 255), 2, f"STOPPED ({duration:.1f}s)"
            else:
                box_color, box_thick, status_lbl = (0, 255, 0), 2, "MOVING"
                
            cv2.rectangle(annotated_frame, (x1, y1), (x2, y2), box_color, box_thick)
            label = f"ID {track_id} | {cls_name} | {status_lbl}"
            (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 2)
            cv2.rectangle(annotated_frame, (x1, y1 - 18), (x1 + w, y1), box_color, -1)
            cv2.putText(annotated_frame, label, (x1, y1 - 4), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 2, lineType=cv2.LINE_AA)
            
            # Draw fade line path
            for h_idx in range(1, len(history)):
                pt1 = (int(history[h_idx - 1][0]), int(history[h_idx - 1][1]))
                pt2 = (int(history[h_idx][0]), int(history[h_idx][1]))
                age_alpha = h_idx / len(history)
                line_color = (int(box_color[0]*age_alpha + 128*(1-age_alpha)), int(box_color[1]*age_alpha + 128*(1-age_alpha)), int(box_color[2]*age_alpha))
                cv2.line(annotated_frame, pt1, pt2, line_color, 2, lineType=cv2.LINE_AA)
                
        if active_hazards_in_frame:
            cv2.rectangle(annotated_frame, (0, 0), (width, 40), (0, 0, 180), -1)
            hazards_desc = ", ".join([f"{name} (ID {tid})" for tid, name in active_hazards_in_frame[:3]])
            cv2.putText(annotated_frame, f"WARNING: ROAD BLOCKED - Active Obstacles: {hazards_desc}", (20, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, lineType=cv2.LINE_AA)
            
        out_writer.write(annotated_frame)
        processed_frames += 1
        if processed_frames % 50 == 0 or processed_frames == frame_count:
            print(f"Processed {processed_frames}/{frame_count} frames")
            
    if out_writer is not None: out_writer.release()
    
    # Export logs
    log_cols = ["timestamp", "event_id", "object_id", "class_name", "confidence_score", "frame_number", "evidence_filename", "video_name", "status"]
    log_df = pd.DataFrame(incident_logs, columns=log_cols)
    csv_path = os.path.join(LOGS_DIR, "incident_log.csv")
    if os.path.exists(csv_path) and os.path.getsize(csv_path) > 0:
        try:
            existing_df = pd.read_csv(csv_path)
            log_df = pd.concat([existing_df, log_df], ignore_index=True)
            log_df = log_df.drop_duplicates(subset=["timestamp", "object_id", "frame_number", "video_name"])
        except Exception: pass
    log_df.to_csv(csv_path, index=False)
    
    avg_conf = log_df['confidence_score'].mean() if len(log_df) > 0 else 0.0
    most_freq_class = log_df['class_name'].mode()[0] if len(log_df) > 0 and not log_df['class_name'].empty else "N/A"
    with open(os.path.join(LOGS_DIR, "summary_stats.json"), 'w') as f:
        json.dump({
            "total_incidents": len(log_df),
            "unique_obstacles_tracked": len(total_tracked_hazards),
            "avg_confidence": float(avg_conf),
            "most_frequent_hazard_class": most_freq_class
        }, f, indent=4)
        
    print(f"Pipeline run complete! Processing speed: {processed_frames / (time.time() - start_time):.2f} FPS")
    display_analytics_dashboard(LOGS_DIR)

# Execute pipeline on the downloaded trash video
run_debris_detection_pipeline(TEST_VIDEO, ANNOTATED_OUT)